# Phase 3 & 4: YOLOv10 Domain-Specific Training & Smart Inference

**Objective:** Train YOLOv10-Medium for table detection on document images, then run inference with intelligent post-processing (box dilation, heuristic filtering).

---

## Table of Contents

### Phase 3: Domain-Specific Training Pipeline
1. [Environment Setup & Installation](#setup)
2. [Configuration](#config)
3. [Custom Augmentation Pipeline](#augmentation)
4. [Model Initialization & Training](#training)
5. [Per-Epoch Metrics](#epoch-metrics)
6. [Training Visualization](#train-viz)
7. [Test Set Evaluation](#test-eval)

### Phase 4: Inference & Smart Post-Processing
8. [Load Best Weights & Run Inference](#inference)
9. [Box Dilation (Padding)](#dilation)
10. [Heuristic Filtering](#filtering)
11. [End-to-End Inference Pipeline](#pipeline)
12. [Post-Processing Evaluation](#post-eval)
13. [Results Visualization](#results-viz)
14. [Export Results](#export)

---

### Key Enhancements Over Standard YOLO Training

| Enhancement | Standard | This Pipeline |
|---|---|---|
| Resolution | 640×640 squish | 1024×1024 letterbox |
| Mosaic | ON | **OFF** (cross-pattern mimics tables) |
| MixUp | ON | **OFF** (ghosting hurts borders) |
| HSV/Color Jitter | ON | **OFF** (structure > color) |
| Flip LR | ON | **OFF** (reading order) |
| CLAHE | OFF | **ON** (contrast on dark scans) |
| Grayscale Aug | OFF | **ON** (learn structure, not color) |
| Micro-rotation | OFF | **ON** (±0.5° realistic scan skew) |
| Post-processing | Raw boxes | Dilation + AR filter + nested removal |

<a id='setup'></a>
## 1. Environment Setup & Installation

In [ ]:
# ============================================================
# Cell 1: Environment Setup & Installation
# ============================================================
import subprocess
import sys

def pip_install(package, flags=None):
    """Install a pip package with optional flags."""
    cmd = [sys.executable, '-m', 'pip', 'install']
    if flags:
        cmd.extend(flags)
    cmd.append(package)
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("="*60)
print(" ENVIRONMENT SETUP")
print("="*60)

# 1. Install YOLOv10 from THU-MIG repo (includes ultralytics fork)
print("\n[1/2] Installing YOLOv10 (THU-MIG)...")
pip_install('git+https://github.com/THU-MIG/yolov10.git', ['-q'])
print("      Done.")

# 2. Install albumentations for custom augmentation pipeline
print("[2/2] Installing albumentations...")
pip_install('albumentations', ['-q', '--upgrade'])
print("      Done.")

# 3. Verify GPU availability
import torch
print(f"\n PyTorch version: {torch.__version__}")
print(f" CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU:             {gpu_name}")
    print(f" VRAM:            {vram_gb:.1f} GB")
else:
    print("  No GPU detected. Training will be extremely slow on CPU!")

# 4. Verify imports
from ultralytics import YOLOv10
import albumentations as A
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, shutil, warnings, time
from collections import defaultdict
from PIL import Image
from matplotlib.patches import Rectangle

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("\n All imports successful.")
print("="*60)

<a id='config'></a>
## 2. Configuration

**Model:** YOLOv10-Medium — better thin-line feature extraction than Small/Nano, fits P100 16GB VRAM.

**Resolution:** 1024×1024 letterbox — documents are usually A4 (portrait). Squishing to square distorts aspect ratio. Letterboxing adds gray padding bars to make the image square without stretching text/lines. 1024px is needed because table borders are often only 1–2 pixels thick; lower resolution makes them disappear.

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================

# =====================================================================
# PATHS  (Kaggle environment — output of Phase 2)
# =====================================================================
DATA_YAML    = '/kaggle/working/yolo_dataset/data.yaml'
IMAGES_DIR   = '/kaggle/working/yolo_dataset/images'
LABELS_DIR   = '/kaggle/working/yolo_dataset/labels'
OUTPUT_DIR   = '/kaggle/working'
CROPS_DIR    = '/kaggle/working/cropped_tables'

# =====================================================================
# MODEL
# =====================================================================
MODEL_VARIANT  = 'yolov10m.pt'     # YOLOv10-Medium
IMGSZ          = 1024              # Letterbox resolution (preserves A4 aspect ratio)

# =====================================================================
# TRAINING HYPERPARAMETERS
# =====================================================================
EPOCHS         = 60                # Max epochs (early stopping may terminate earlier)
PATIENCE       = 10                # Early stopping patience (epochs without improvement)
BATCH_SIZE     = 8                 # Conservative for P100 16GB at 1024px (~12-14GB VRAM)
OPTIMIZER      = 'AdamW'           # Better generalization than SGD for small datasets
LR0            = 1e-3              # Initial learning rate
LRF            = 0.01              # Final LR factor (cosine decay: LR0 * LRF = 1e-5)
WEIGHT_DECAY   = 5e-4              # AdamW L2 regularization
WARMUP_EPOCHS  = 3                 # Gradual LR ramp-up
LABEL_SMOOTHING = 0.01             # Slight smoothing for noisy annotations
BOX_LOSS_GAIN  = 7.5               # Increase box loss weight (precise localization critical)

# =====================================================================
# AUGMENTATION (Document-Specific Strategy)
# =====================================================================
# DISABLED — harmful for document analysis
AUG_MOSAIC     = 0.0   # OFF: cross-pattern mimics table intersections → false positives
AUG_MIXUP      = 0.0   # OFF: ghosting damages border detection
AUG_HSV_H      = 0.0   # OFF: documents are structure-based, not color-based
AUG_HSV_S      = 0.0   # OFF
AUG_HSV_V      = 0.0   # OFF
AUG_FLIPLR     = 0.0   # OFF: preserve text reading order
AUG_FLIPUD     = 0.0   # OFF: unrealistic for documents

# ENABLED — beneficial for document analysis
AUG_DEGREES    = 0.5   # ±0.5° micro-rotation (realistic scanner skew)
AUG_SCALE      = 0.2   # ±20% mild scale jitter

# CLAHE / Grayscale — injected via Albumentations (see Cell 3)
AUG_CLAHE_P    = 0.3   # 30% chance of CLAHE (contrast enhancement for dark scans)
AUG_GRAY_P     = 0.2   # 20% chance of grayscale (forces structural learning)

# =====================================================================
# INFERENCE & POST-PROCESSING (Phase 4)
# =====================================================================
CONF_THRESH    = 0.25              # Confidence threshold
BOX_PADDING_PX = 15                # Dilation padding in pixels
AR_MIN         = 0.1               # Min aspect ratio (w/h) — discard < 1:10
AR_MAX         = 10.0              # Max aspect ratio (w/h) — discard > 10:1
MIN_AREA_PCT   = 0.01              # Min box area as % of image (discard < 1%)
NESTED_IOU     = 0.85              # IoU threshold for nested box detection
NMS_IOU_THRESH = 0.5               # Light NMS for duplicate safety check

# =====================================================================
# REPRODUCIBILITY
# =====================================================================
RANDOM_SEED    = 42

# =====================================================================
# CLASS INFO
# =====================================================================
CLASS_NAMES    = ['table']
NC             = 1

print("="*60)
print(" CONFIGURATION SUMMARY")
print("="*60)
print(f"\n Model:      {MODEL_VARIANT}")
print(f" Resolution: {IMGSZ}x{IMGSZ} (letterbox)")
print(f" Epochs:     {EPOCHS} (patience={PATIENCE})")
print(f" Batch Size: {BATCH_SIZE}")
print(f" Optimizer:  {OPTIMIZER} (lr={LR0}, wd={WEIGHT_DECAY})")
print(f" Box Loss:   {BOX_LOSS_GAIN}")
print(f"\n Augmentation Strategy:")
print(f"   Mosaic:  {AUG_MOSAIC}  (OFF)")
print(f"   MixUp:   {AUG_MIXUP}  (OFF)")
print(f"   HSV:     {AUG_HSV_H}/{AUG_HSV_S}/{AUG_HSV_V}  (OFF)")
print(f"   Flip:    LR={AUG_FLIPLR}, UD={AUG_FLIPUD}  (OFF)")
print(f"   Rotate:  ±{AUG_DEGREES}°  (ON)")
print(f"   Scale:   ±{AUG_SCALE*100:.0f}%  (ON)")
print(f"   CLAHE:   p={AUG_CLAHE_P}  (ON)")
print(f"   Gray:    p={AUG_GRAY_P}  (ON)")
print(f"\n Post-Processing:")
print(f"   Confidence:  {CONF_THRESH}")
print(f"   Box Padding: {BOX_PADDING_PX}px")
print(f"   AR Range:    [{AR_MIN}, {AR_MAX}]")
print(f"   Min Area:    {MIN_AREA_PCT*100:.0f}% of image")
print(f"\n Data YAML:  {DATA_YAML}")
print("="*60)

<a id='augmentation'></a>
## 3. Custom Augmentation Pipeline (Albumentations)

**Strategy:** Monkey-patch the ultralytics `Albumentations` wrapper class to inject document-specific augmentations:
- **CLAHE** (`p=0.3`): Contrast Limited Adaptive Histogram Equalization — improves contrast on scans with dark/shadowy backgrounds
- **Grayscale** (`p=0.2`): Forces the model to learn structural features (lines, borders) instead of relying on header colors

The built-in harmful augmentations (mosaic, mixup, HSV, flips) are disabled via `model.train()` parameters.

In [ ]:
# ============================================================
# Cell 3: Custom Augmentation Pipeline — Albumentations Injection
# ============================================================

import albumentations as A
from ultralytics.data import augment as ultralytics_augment

print("="*60)
print(" CUSTOM AUGMENTATION PIPELINE")
print("="*60)

def _patched_albumentations_init(self, p=1.0):
    """
    Replace the default Albumentations pipeline with document-specific transforms.
    
    Injected transforms:
    - CLAHE: Contrast enhancement for dark/shadowy scans
    - ToGray: Force structural learning (lines/borders over colors)
    
    All other ultralytics augmentations (mosaic, mixup, HSV, flip)
    are controlled via model.train() parameters — not here.
    """
    self.p = p
    self.transform = None

    try:
        transforms = [
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=AUG_CLAHE_P),
            A.ToGray(p=AUG_GRAY_P),
        ]

        self.transform = A.Compose(
            transforms,
            bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'])
        )

        transform_names = ', '.join(t.__class__.__name__ for t in transforms)
        print(f"   Albumentations pipeline injected: [{transform_names}]")

    except Exception as e:
        print(f"   Albumentations injection failed: {e}")
        self.transform = None

# Apply the monkey-patch
ultralytics_augment.Albumentations.__init__ = _patched_albumentations_init

print("\n Monkey-patch applied to ultralytics.data.augment.Albumentations")
print(" The following augmentations will be applied during training:")
print(f"   - CLAHE (clip_limit=2.0, p={AUG_CLAHE_P})")
print(f"   - ToGray (p={AUG_GRAY_P})")
print("\n The following augmentations are DISABLED via model.train() params:")
print("   - Mosaic (cross-pattern mimics table intersections)")
print("   - MixUp (ghosting damages border detection)")
print("   - HSV/Color Jitter (documents are structure-based)")
print("   - Horizontal/Vertical Flip (text reading order)")
print("="*60)

<a id='training'></a>
## 4. Model Initialization & Training

**Model:** YOLOv10-M (Medium) — fits P100 16GB VRAM, superior feature extraction for thin grid lines compared to Nano/Small variants.

**Training Strategy:**
- **AdamW** optimizer with cosine LR decay ($10^{-3} \to 10^{-5}$)
- **3-epoch warmup** for gradient stabilization
- **Early stopping** (patience=10) to prevent overfitting on ~1500 samples
- **Box loss gain = 7.5** (default is 7.5) — precise table localization is paramount
- **Label smoothing = 0.01** — accounts for annotation noise found in Phase 1

In [ ]:
# ============================================================
# Cell 4: Model Initialization & Training
# ============================================================

print("="*60)
print(" MODEL INITIALIZATION & TRAINING")
print("="*60)

# -------------------- Download Weights --------------------
import urllib.request
import torch

WEIGHTS_URL = 'https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10m.pt'
if not os.path.exists(MODEL_VARIANT):
    print(f"\n Downloading {MODEL_VARIANT} from {WEIGHTS_URL}...")
    try:
        urllib.request.urlretrieve(WEIGHTS_URL, MODEL_VARIANT)
        print("   Download complete.")
    except Exception as e:
        print(f"   Download failed: {e}")
        print("   Please manually download the weights or check internet connection.")
else:
    print(f"\n Found local weights: {MODEL_VARIANT}")

# -------------------- Fix PyTorch 2.6+ Safe Load Issue --------------------
# YOLOv10 uses a custom class that isn't in PyTorch's default safe globals list.
# We explicitly allow it to avoid UnpicklingError.
try:
    from ultralytics.nn.tasks import YOLOv10DetectionModel
    torch.serialization.add_safe_globals([YOLOv10DetectionModel])
    print("   Added YOLOv10DetectionModel to torch safe globals.")
except ImportError:
    print("   Could not import YOLOv10DetectionModel. Attempting fallback monkey-patch...")
    # Fallback: Temporarily disable weights_only enforcement if the above fails
    _original_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        return _original_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    print("   Monkey-patched torch.load to set weights_only=False.")

# -------------------- Initialize Model --------------------
print(f"\n Loading pretrained {MODEL_VARIANT}...")
if os.path.exists(MODEL_VARIANT):
    model = YOLOv10(MODEL_VARIANT)
    print(f"   Model loaded: YOLOv10-Medium")
    print(f"   Pretrained on COCO — fine-tuning for table detection")
else:
    raise FileNotFoundError(f"Weights file {MODEL_VARIANT} not found. Download failed.")

# -------------------- Verify data.yaml --------------------
assert os.path.exists(DATA_YAML), f"data.yaml not found at {DATA_YAML}"
print(f"   data.yaml verified: {DATA_YAML}")

# -------------------- Train --------------------
print(f"\n Starting training...")
print(f"   Epochs:     {EPOCHS} (early stopping patience={PATIENCE})")
print(f"   Resolution: {IMGSZ}x{IMGSZ}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Optimizer:  {OPTIMIZER}")
print("-"*60)

train_start = time.time()

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    
    # Optimizer
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    
    # Loss weights
    box=BOX_LOSS_GAIN,
    label_smoothing=LABEL_SMOOTHING,
    
    # Augmentation — DISABLED (document-specific)
    mosaic=AUG_MOSAIC,
    mixup=AUG_MIXUP,
    hsv_h=AUG_HSV_H,
    hsv_s=AUG_HSV_S,
    hsv_v=AUG_HSV_V,
    fliplr=AUG_FLIPLR,
    flipud=AUG_FLIPUD,
    
    # Augmentation — ENABLED
    degrees=AUG_DEGREES,
    scale=AUG_SCALE,
    
    # Mosaic close epoch (already disabled, but set to 0 for safety)
    close_mosaic=0,
    
    # Early stopping
    patience=PATIENCE,
    
    # Reproducibility
    seed=RANDOM_SEED,
    deterministic=True,
    
    # Output
    project=os.path.join(OUTPUT_DIR, 'runs', 'detect'),
    name='table_detection',
    exist_ok=True,
    verbose=True,
    plots=True,
    save=True,
    val=True,
)

train_elapsed = time.time() - train_start

# -------------------- Identify output paths --------------------
TRAIN_DIR = os.path.join(OUTPUT_DIR, 'runs', 'detect', 'table_detection')
BEST_WEIGHTS = os.path.join(TRAIN_DIR, 'weights', 'best.pt')
LAST_WEIGHTS = os.path.join(TRAIN_DIR, 'weights', 'last.pt')
RESULTS_CSV = os.path.join(TRAIN_DIR, 'results.csv')

print(f"\n Training complete in {train_elapsed/60:.1f} minutes.")
print(f"   Best weights: {BEST_WEIGHTS}")
print(f"   Last weights: {LAST_WEIGHTS}")
print(f"   Results CSV:  {RESULTS_CSV}")

<a id='epoch-metrics'></a>
## 5. Per-Epoch Metrics

Parse `results.csv` from the training run and display a formatted table with all metrics per epoch. Compute and append the F1-Score:

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
# ============================================================
# Cell 5: Per-Epoch Metrics Display
# ============================================================

print("="*60)
print(" PER-EPOCH TRAINING METRICS")
print("="*60)

# -------------------- Load results.csv --------------------
assert os.path.exists(RESULTS_CSV), f"Results CSV not found: {RESULTS_CSV}"

df_results = pd.read_csv(RESULTS_CSV)
df_results.columns = df_results.columns.str.strip()  # Clean whitespace from column names

print(f"\n   Loaded {len(df_results)} epochs from results.csv")
print(f"   Columns: {list(df_results.columns)}")

# -------------------- Rename columns for readability --------------------
# Ultralytics results.csv has columns like:
#   epoch, train/box_loss, train/cls_loss, train/dfl_loss,
#   metrics/precision(B), metrics/recall(B), metrics/mAP50(B), metrics/mAP50-95(B),
#   val/box_loss, val/cls_loss, val/dfl_loss, lr/pg0, lr/pg1, lr/pg2

col_map = {}
for col in df_results.columns:
    c = col.strip()
    if c == 'epoch':
        col_map[col] = 'Epoch'
    elif 'train/box_loss' in c:
        col_map[col] = 'Train Box Loss'
    elif 'train/cls_loss' in c:
        col_map[col] = 'Train Cls Loss'
    elif 'train/dfl_loss' in c:
        col_map[col] = 'Train DFL Loss'
    elif 'val/box_loss' in c:
        col_map[col] = 'Val Box Loss'
    elif 'val/cls_loss' in c:
        col_map[col] = 'Val Cls Loss'
    elif 'val/dfl_loss' in c:
        col_map[col] = 'Val DFL Loss'
    elif 'precision' in c.lower():
        col_map[col] = 'Precision'
    elif 'recall' in c.lower():
        col_map[col] = 'Recall'
    elif 'mAP50-95' in c or 'mAP50_95' in c:
        col_map[col] = 'mAP@0.5:0.95'
    elif 'mAP50' in c:
        col_map[col] = 'mAP@0.5'
    elif 'lr/pg0' in c:
        col_map[col] = 'LR'

df_metrics = df_results.rename(columns=col_map)

# -------------------- Compute F1-Score --------------------
if 'Precision' in df_metrics.columns and 'Recall' in df_metrics.columns:
    P = df_metrics['Precision']
    R = df_metrics['Recall']
    df_metrics['F1-Score'] = np.where(
        (P + R) > 0,
        2 * P * R / (P + R),
        0.0
    )

# -------------------- Epoch column as int --------------------
if 'Epoch' in df_metrics.columns:
    df_metrics['Epoch'] = df_metrics['Epoch'].astype(int)

# -------------------- Select display columns --------------------
display_cols = [c for c in [
    'Epoch', 'Train Box Loss', 'Train Cls Loss', 'Train DFL Loss',
    'Val Box Loss', 'Val Cls Loss', 'Val DFL Loss',
    'Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95', 'F1-Score', 'LR'
] if c in df_metrics.columns]

df_display = df_metrics[display_cols].copy()

# -------------------- Display full table --------------------
print("\n" + "-"*60)
print(" FULL METRICS TABLE")
print("-"*60)

# Format numeric columns
fmt_cols = [c for c in display_cols if c != 'Epoch']
styled = df_display.style.format(
    {c: '{:.4f}' for c in fmt_cols}
).hide(axis='index')

display(styled)

# -------------------- Best Epoch --------------------
if 'mAP@0.5:0.95' in df_metrics.columns:
    best_idx = df_metrics['mAP@0.5:0.95'].idxmax()
    best_epoch = df_metrics.loc[best_idx]
    
    print("\n" + "-"*60)
    print(" BEST EPOCH (by mAP@0.5:0.95)")
    print("-"*60)
    print(f"   Epoch:         {int(best_epoch.get('Epoch', best_idx))}")
    print(f"   Precision:     {best_epoch.get('Precision', 0):.4f}")
    print(f"   Recall:        {best_epoch.get('Recall', 0):.4f}")
    print(f"   mAP@0.5:       {best_epoch.get('mAP@0.5', 0):.4f}")
    print(f"   mAP@0.5:0.95:  {best_epoch.get('mAP@0.5:0.95', 0):.4f}")
    print(f"   F1-Score:      {best_epoch.get('F1-Score', 0):.4f}")

# -------------------- Early Stopping Info --------------------
actual_epochs = len(df_metrics)
print(f"\n   Training ran for {actual_epochs} epochs (max={EPOCHS}, patience={PATIENCE})")
if actual_epochs < EPOCHS:
    print(f"    Early stopping triggered at epoch {actual_epochs}")
else:
    print(f"   Completed all {EPOCHS} epochs (no early stop)")

print("="*60)

<a id='train-viz'></a>
## 6. Training Visualization

Four-panel visualization of training dynamics:
1. **Loss Curves** — Train vs Val for Box/Cls/DFL losses (convergence check)
2. **Detection Metrics** — Precision, Recall, mAP@0.5, mAP@0.5:0.95 over epochs
3. **F1-Score Curve** — Harmonic mean of Precision and Recall
4. **Learning Rate Schedule** — Warmup + cosine decay verification

In [ ]:
# ============================================================
# Cell 6: Training Visualization
# ============================================================

print("="*60)
print(" TRAINING VISUALIZATION")
print("="*60)

epochs = df_metrics['Epoch'] if 'Epoch' in df_metrics.columns else range(len(df_metrics))

# =====================================================================
# PLOT 1: Loss Curves (2x2 grid — Train vs Val for Box/Cls/DFL + Combined)
# =====================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box Loss
ax = axes[0, 0]
if 'Train Box Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Train Box Loss'], 'b-', linewidth=2, label='Train', alpha=0.8)
if 'Val Box Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Val Box Loss'], 'r--', linewidth=2, label='Val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Box Loss', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Cls Loss
ax = axes[0, 1]
if 'Train Cls Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Train Cls Loss'], 'b-', linewidth=2, label='Train', alpha=0.8)
if 'Val Cls Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Val Cls Loss'], 'r--', linewidth=2, label='Val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Classification Loss', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# DFL Loss
ax = axes[1, 0]
if 'Train DFL Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Train DFL Loss'], 'b-', linewidth=2, label='Train', alpha=0.8)
if 'Val DFL Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Val DFL Loss'], 'r--', linewidth=2, label='Val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('DFL Loss (Distribution Focal Loss)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Combined Total Loss
ax = axes[1, 1]
train_losses = []
val_losses = []
for prefix, target in [('Train', train_losses), ('Val', val_losses)]:
    total = None
    for loss_type in ['Box Loss', 'Cls Loss', 'DFL Loss']:
        col = f'{prefix} {loss_type}'
        if col in df_metrics.columns:
            total = df_metrics[col] if total is None else total + df_metrics[col]
    if total is not None:
        target.extend(total.tolist())
        style = 'b-' if prefix == 'Train' else 'r--'
        ax.plot(epochs, total, style, linewidth=2, label=f'{prefix} Total', alpha=0.8)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Total Loss (Box + Cls + DFL)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('Training Loss Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# =====================================================================
# PLOT 2: Detection Metrics Over Epochs
# =====================================================================
fig, ax = plt.subplots(figsize=(12, 5))

metric_configs = [
    ('Precision', '#E74C3C', '-'),
    ('Recall', '#3498DB', '-'),
    ('mAP@0.5', '#2ECC71', '-'),
    ('mAP@0.5:0.95', '#9B59B6', '--'),
]

for col_name, color, style in metric_configs:
    if col_name in df_metrics.columns:
        ax.plot(epochs, df_metrics[col_name], color=color, linestyle=style,
                linewidth=2, label=col_name, alpha=0.85)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Detection Metrics Over Epochs', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =====================================================================
# PLOT 3: F1-Score Curve
# =====================================================================
if 'F1-Score' in df_metrics.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(epochs, df_metrics['F1-Score'], color='#F39C12', linewidth=2.5, 
            label='F1-Score', alpha=0.9)
    ax.fill_between(epochs, df_metrics['F1-Score'], alpha=0.15, color='#F39C12')
    
    # Mark best F1
    best_f1_idx = df_metrics['F1-Score'].idxmax()
    best_f1_epoch = epochs.iloc[best_f1_idx] if hasattr(epochs, 'iloc') else best_f1_idx
    best_f1_val = df_metrics['F1-Score'].iloc[best_f1_idx]
    ax.axvline(x=best_f1_epoch, color='red', linestyle=':', alpha=0.5)
    ax.scatter([best_f1_epoch], [best_f1_val], color='red', s=100, zorder=5, 
               label=f'Best F1={best_f1_val:.4f} (Epoch {int(best_f1_epoch)})')
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title('F1-Score Over Epochs', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# =====================================================================
# PLOT 4: Learning Rate Schedule
# =====================================================================
if 'LR' in df_metrics.columns:
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(epochs, df_metrics['LR'], color='#1ABC9C', linewidth=2, label='Learning Rate (pg0)')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Learning Rate', fontsize=12)
    ax.set_title('Learning Rate Schedule (Warmup + Cosine Decay)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(-4, -4))
    plt.tight_layout()
    plt.show()

print("   All training plots generated.")
print("="*60)

<a id='test-eval'></a>
## 7. Test Set Evaluation

Evaluate the best model checkpoint on the held-out **test split** (5% of data, document-grouped — zero leakage from train/val).

**Metrics:**
- **IoU** (Intersection over Union) — localization quality
- **mAP@0.5** and **mAP@0.5:0.95** — detection accuracy
- **Precision** and **Recall** — completeness vs. correctness
- **F1-Score** — harmonic mean of P and R

In [ ]:
# ============================================================
# Cell 7: Test Set Evaluation
# ============================================================

print("="*60)
print(" TEST SET EVALUATION (best.pt)")
print("="*60)

# -------------------- Load Best Weights --------------------
print(f"\n Loading best weights: {BEST_WEIGHTS}")
model_eval = YOLOv10(BEST_WEIGHTS)

# -------------------- Run Validation on Test Split --------------------
print(f" Evaluating on test split...")

test_results = model_eval.val(
    data=DATA_YAML,
    split='test',
    imgsz=IMGSZ,
    conf=CONF_THRESH,
    batch=BATCH_SIZE,
    plots=True,
    save_json=True,
    verbose=True,
)

# -------------------- Extract Metrics --------------------
# test_results.box contains: mp, mr, map50, map (map50-95)
test_precision = test_results.box.mp       # Mean Precision
test_recall    = test_results.box.mr       # Mean Recall
test_map50     = test_results.box.map50    # mAP@0.5
test_map5095   = test_results.box.map      # mAP@0.5:0.95

# F1-Score
if (test_precision + test_recall) > 0:
    test_f1 = 2 * test_precision * test_recall / (test_precision + test_recall)
else:
    test_f1 = 0.0

# -------------------- Display Results --------------------
print("\n" + "-"*60)
print(" TEST SET RESULTS")
print("-"*60)

test_summary = [
    ['Precision',      f'{test_precision:.4f}'],
    ['Recall',         f'{test_recall:.4f}'],
    ['F1-Score',       f'{test_f1:.4f}'],
    ['mAP@0.5',       f'{test_map50:.4f}'],
    ['mAP@0.5:0.95',  f'{test_map5095:.4f}'],
]

df_test_summary = pd.DataFrame(test_summary, columns=['Metric', 'Value'])
display(df_test_summary.style.hide(axis='index'))

# -------------------- Interpretation --------------------
print("\n INTERPRETATION:")
if test_map50 >= 0.90:
    print(f"    Excellent: mAP@0.5 = {test_map50:.4f} (>= 0.90)")
elif test_map50 >= 0.80:
    print(f"    Good: mAP@0.5 = {test_map50:.4f} (>= 0.80)")
elif test_map50 >= 0.70:
    print(f"    Fair: mAP@0.5 = {test_map50:.4f} (>= 0.70)")
else:
    print(f"    Needs improvement: mAP@0.5 = {test_map50:.4f} (< 0.70)")

if test_f1 >= 0.85:
    print(f"    F1-Score {test_f1:.4f} indicates strong balance between precision and recall")
elif test_f1 >= 0.70:
    print(f"    F1-Score {test_f1:.4f} is acceptable but may benefit from threshold tuning")
else:
    print(f"    F1-Score {test_f1:.4f} suggests a precision-recall imbalance — check conf threshold")

# -------------------- Display Built-in Plots --------------------
# Ultralytics generates PR curves, confusion matrix, etc. in the val output dir
print("\n Built-in evaluation plots saved to training directory.")
print("   Check for: PR_curve.png, confusion_matrix.png, F1_curve.png")

# Try to display the PR curve if it exists
pr_curve_path = os.path.join(TRAIN_DIR, 'PR_curve.png')
f1_curve_path = os.path.join(TRAIN_DIR, 'F1_curve.png')
cm_path = os.path.join(TRAIN_DIR, 'confusion_matrix.png')

available_plots = []
for path, name in [(pr_curve_path, 'PR Curve'), (f1_curve_path, 'F1 Curve'), (cm_path, 'Confusion Matrix')]:
    if os.path.exists(path):
        available_plots.append((path, name))

if available_plots:
    n_plots = len(available_plots)
    fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    
    for ax, (path, name) in zip(axes, available_plots):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("="*60)

---

# Phase 4: Inference & Smart Post-Processing

**Objective:** Go beyond raw `model.predict()` — apply intelligent post-processing to improve crop quality for downstream table structure recognition.

**Pipeline:** Raw Predictions → Box Dilation → Heuristic Filtering → Final Crops

---

<a id='inference'></a>
## 8. Load Best Weights & Run Inference

In [ ]:
# ============================================================
# Cell 8: Load Best Weights & Run Inference on Test Set
# ============================================================

print("="*60)
print(" INFERENCE ON TEST SET")
print("="*60)

# -------------------- Load Best Model --------------------
print(f"\n Loading best weights: {BEST_WEIGHTS}")
model_infer = YOLOv10(BEST_WEIGHTS)

# -------------------- Collect Test Images --------------------
test_images_dir = os.path.join(IMAGES_DIR, 'test')
test_image_files = sorted([
    f for f in os.listdir(test_images_dir) 
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f"   Found {len(test_image_files)} test images in {test_images_dir}")

# -------------------- Run Inference --------------------
print(f"\n Running inference (conf={CONF_THRESH}, imgsz={IMGSZ})...")
print(f"   YOLOv10 uses NMS-free dual-head design — no explicit NMS needed")
print(f"   Adding light IoU-based duplicate check (iou_thresh={NMS_IOU_THRESH}) as safety net")

inference_start = time.time()

raw_predictions = {}  # {filename: list of {box, conf, cls}}

for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)
    
    # Run prediction
    preds = model_infer.predict(
        source=img_path,
        conf=CONF_THRESH,
        imgsz=IMGSZ,
        save=False,
        verbose=False,
    )
    
    # Extract results
    result = preds[0]
    boxes = result.boxes
    
    detections = []
    if len(boxes) > 0:
        xyxy   = boxes.xyxy.cpu().numpy()    # [x1, y1, x2, y2]
        confs  = boxes.conf.cpu().numpy()     # confidence scores
        clses  = boxes.cls.cpu().numpy()      # class IDs
        
        for i in range(len(xyxy)):
            detections.append({
                'box': xyxy[i].tolist(),      # [x1, y1, x2, y2]
                'conf': float(confs[i]),
                'cls': int(clses[i]),
            })
    
    raw_predictions[img_file] = detections

inference_elapsed = time.time() - inference_start

# -------------------- Summary --------------------
total_detections = sum(len(v) for v in raw_predictions.values())
images_with_dets = sum(1 for v in raw_predictions.values() if len(v) > 0)
images_no_dets   = len(raw_predictions) - images_with_dets

print(f"\n" + "-"*60)
print(f" RAW INFERENCE RESULTS")
print(f"-"*60)
print(f"   Total images processed:      {len(raw_predictions)}")
print(f"   Images with detections:       {images_with_dets}")
print(f"   Images without detections:    {images_no_dets}")
print(f"   Total raw detections:         {total_detections}")
print(f"   Avg detections per image:     {total_detections/max(len(raw_predictions),1):.2f}")
print(f"   Inference time:               {inference_elapsed:.1f}s ({inference_elapsed/max(len(raw_predictions),1)*1000:.0f}ms/image)")

# -------------------- Confidence Distribution --------------------
all_confs = [d['conf'] for dets in raw_predictions.values() for d in dets]
if all_confs:
    print(f"\n   Confidence Stats:")
    print(f"      Min:    {min(all_confs):.4f}")
    print(f"      Max:    {max(all_confs):.4f}")
    print(f"      Mean:   {np.mean(all_confs):.4f}")
    print(f"      Median: {np.median(all_confs):.4f}")

print("="*60)

<a id='dilation'></a>
## 9. Box Dilation (Padding)

**The Critical Addition:** Raw detections often cut precisely on the table border line, destroying it during cropping. This makes downstream table structure recognition (row/column extraction) fail.

**Solution:** Expand every predicted bounding box outward by a fixed margin (default: 15 pixels) before cropping, ensuring the crop includes:
- The entire black border of the table
- A tiny whitespace buffer around it

Boundary clamping prevents the expanded box from exceeding image dimensions.

In [ ]:
# ============================================================
# Cell 9: Box Dilation (Padding) Function
# ============================================================

print("="*60)
print(" BOX DILATION (PADDING)")
print("="*60)

def dilate_boxes(detections, padding_px, img_w, img_h):
    """
    Expand bounding boxes outward by a fixed pixel margin.
    
    Args:
        detections: list of dicts with 'box' key [x1, y1, x2, y2]
        padding_px: number of pixels to expand on each side
        img_w: image width (for clamping)
        img_h: image height (for clamping)
    
    Returns:
        list of dicts with dilated boxes (new list, original unchanged)
    """
    dilated = []
    for det in detections:
        x1, y1, x2, y2 = det['box']
        
        # Expand outward
        x1_new = max(0, x1 - padding_px)
        y1_new = max(0, y1 - padding_px)
        x2_new = min(img_w, x2 + padding_px)
        y2_new = min(img_h, y2 + padding_px)
        
        dilated.append({
            'box': [x1_new, y1_new, x2_new, y2_new],
            'conf': det['conf'],
            'cls': det['cls'],
            'box_raw': det['box'],  # Keep original for comparison
        })
    
    return dilated

# -------------------- Apply Dilation to All Predictions --------------------
dilated_predictions = {}

for img_file, detections in raw_predictions.items():
    if not detections:
        dilated_predictions[img_file] = []
        continue
    
    # Get image dimensions
    img_path = os.path.join(test_images_dir, img_file)
    img = Image.open(img_path)
    img_w, img_h = img.size
    
    dilated_predictions[img_file] = dilate_boxes(detections, BOX_PADDING_PX, img_w, img_h)

# -------------------- Report Dilation Effect --------------------
total_expansion = []
for img_file, dilated in dilated_predictions.items():
    for det in dilated:
        if 'box_raw' in det:
            raw = det['box_raw']
            dil = det['box']
            raw_area = (raw[2] - raw[0]) * (raw[3] - raw[1])
            dil_area = (dil[2] - dil[0]) * (dil[3] - dil[1])
            if raw_area > 0:
                expansion_pct = (dil_area - raw_area) / raw_area * 100
                total_expansion.append(expansion_pct)

if total_expansion:
    print(f"\n Dilation Applied: {BOX_PADDING_PX}px on each side")
    print(f"   Boxes dilated:           {len(total_expansion)}")
    print(f"   Mean area expansion:     {np.mean(total_expansion):.1f}%")
    print(f"   Median area expansion:   {np.median(total_expansion):.1f}%")
    print(f"   Min / Max expansion:     {min(total_expansion):.1f}% / {max(total_expansion):.1f}%")
else:
    print("\n   No detections to dilate.")

print("="*60)

<a id='filtering'></a>
## 10. Heuristic Filtering

Three filters to remove false positives and redundant detections:

1. **Aspect Ratio Filter:** Discard boxes with $\text{AR} = w/h < 0.1$ or $> 10.0$ — real tables rarely have extreme proportions
2. **Nested Box Removal:** If box A is fully contained inside box B, remove A — fixes nested false positives
3. **Minimum Area Filter:** Discard boxes smaller than 1% of the image area — likely noise

In [ ]:
# ============================================================
# Cell 10: Heuristic Filtering
# ============================================================

print("="*60)
print(" HEURISTIC FILTERING")
print("="*60)

def compute_iou_xyxy(box1, box2):
    """Compute IoU between two boxes in [x1, y1, x2, y2] format."""
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0

def compute_containment(inner_box, outer_box):
    """
    Compute what fraction of inner_box's area is inside outer_box.
    Returns 1.0 if inner is fully contained in outer.
    """
    inter_x1 = max(inner_box[0], outer_box[0])
    inter_y1 = max(inner_box[1], outer_box[1])
    inter_x2 = min(inner_box[2], outer_box[2])
    inter_y2 = min(inner_box[3], outer_box[3])
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    inner_area = (inner_box[2] - inner_box[0]) * (inner_box[3] - inner_box[1])
    
    return inter_area / inner_area if inner_area > 0 else 0.0

def filter_detections(detections, img_w, img_h, 
                      ar_min=AR_MIN, ar_max=AR_MAX,
                      min_area_pct=MIN_AREA_PCT,
                      nested_iou_thresh=NESTED_IOU):
    """
    Apply heuristic filters to detections.
    
    Filters:
    1. Aspect Ratio: remove boxes with AR < ar_min or AR > ar_max
    2. Minimum Area: remove boxes smaller than min_area_pct of image
    3. Nested Box Removal: if box A is fully inside box B, remove A
    
    Args:
        detections: list of dicts with 'box' [x1, y1, x2, y2], 'conf', 'cls'
        img_w, img_h: image dimensions
        ar_min, ar_max: aspect ratio range
        min_area_pct: minimum box area as fraction of image area
        nested_iou_thresh: containment threshold for nested removal
    
    Returns:
        filtered: list of surviving detections
        filter_log: dict with counts of removed boxes per filter
    """
    filter_log = {
        'input': len(detections),
        'ar_removed': 0,
        'area_removed': 0,
        'nested_removed': 0,
    }
    
    if not detections:
        filter_log['output'] = 0
        return [], filter_log
    
    img_area = img_w * img_h
    surviving = []
    
    # ---- Filter 1: Aspect Ratio ----
    for det in detections:
        x1, y1, x2, y2 = det['box']
        w = x2 - x1
        h = y2 - y1
        
        if h <= 0 or w <= 0:
            filter_log['ar_removed'] += 1
            continue
        
        ar = w / h
        if ar < ar_min or ar > ar_max:
            filter_log['ar_removed'] += 1
            continue
        
        surviving.append(det)
    
    # ---- Filter 2: Minimum Area ----
    area_filtered = []
    for det in surviving:
        x1, y1, x2, y2 = det['box']
        box_area = (x2 - x1) * (y2 - y1)
        
        if box_area / img_area < min_area_pct:
            filter_log['area_removed'] += 1
            continue
        
        area_filtered.append(det)
    
    surviving = area_filtered
    
    # ---- Filter 3: Nested Box Removal ----
    # For every pair, if box A is fully contained inside box B, remove A
    if len(surviving) > 1:
        keep_mask = [True] * len(surviving)
        
        for i in range(len(surviving)):
            if not keep_mask[i]:
                continue
            for j in range(len(surviving)):
                if i == j or not keep_mask[j]:
                    continue
                
                # Check if box j is contained inside box i
                containment = compute_containment(surviving[j]['box'], surviving[i]['box'])
                if containment >= nested_iou_thresh:
                    # j is inside i — remove j (the inner/smaller one)
                    keep_mask[j] = False
                    filter_log['nested_removed'] += 1
        
        surviving = [det for det, keep in zip(surviving, keep_mask) if keep]
    
    filter_log['output'] = len(surviving)
    return surviving, filter_log

# -------------------- Apply Filters to All Dilated Predictions --------------------
filtered_predictions = {}
total_filter_log = defaultdict(int)

for img_file, detections in dilated_predictions.items():
    if not detections:
        filtered_predictions[img_file] = []
        continue
    
    # Get image dimensions
    img_path = os.path.join(test_images_dir, img_file)
    img = Image.open(img_path)
    img_w, img_h = img.size
    
    filtered, log = filter_detections(detections, img_w, img_h)
    filtered_predictions[img_file] = filtered
    
    for key, val in log.items():
        total_filter_log[key] += val

# -------------------- Report --------------------
print(f"\n" + "-"*60)
print(f" FILTERING SUMMARY")
print(f"-"*60)
print(f"   Input detections:          {total_filter_log['input']}")
print(f"   Removed by AR filter:      {total_filter_log['ar_removed']}  (AR<{AR_MIN} or AR>{AR_MAX})")
print(f"   Removed by area filter:    {total_filter_log['area_removed']}  (area<{MIN_AREA_PCT*100:.0f}% of image)")
print(f"   Removed by nested filter:  {total_filter_log['nested_removed']}  (fully inside another box)")
print(f"   Output detections:         {total_filter_log['output']}")

total_removed = total_filter_log['input'] - total_filter_log['output']
if total_filter_log['input'] > 0:
    print(f"\n   Total removed: {total_removed} ({total_removed/total_filter_log['input']*100:.1f}%)")
else:
    print(f"\n   No detections to filter.")

print("="*60)

<a id='pipeline'></a>
## 11. End-to-End Inference Pipeline

Unified pipeline function that chains: **Predict → Dilate → Filter → Crop**

In [ ]:
# ============================================================
# Cell 11: End-to-End Inference Pipeline
# ============================================================

print("="*60)
print(" END-TO-END INFERENCE PIPELINE")
print("="*60)

def run_pipeline(image_path, model, conf=CONF_THRESH, padding=BOX_PADDING_PX,
                 ar_min=AR_MIN, ar_max=AR_MAX, min_area_pct=MIN_AREA_PCT,
                 nested_iou=NESTED_IOU):
    """
    Full inference pipeline: Predict → Dilate → Filter → Crop
    
    Args:
        image_path: path to input image
        model: loaded YOLOv10 model
        conf: confidence threshold
        padding: box dilation pixels
        ar_min, ar_max: aspect ratio bounds
        min_area_pct: minimum area filter
        nested_iou: containment threshold
    
    Returns:
        dict with keys:
            - 'image_path': input path
            - 'img_w', 'img_h': dimensions
            - 'raw_detections': list of raw boxes
            - 'dilated_detections': list after dilation
            - 'filtered_detections': list after filtering
            - 'crops': list of PIL Image crops
            - 'filter_log': filtering statistics
    """
    # Load image
    img = Image.open(image_path)
    img_w, img_h = img.size
    
    # Step 1: Predict
    preds = model.predict(source=image_path, conf=conf, imgsz=IMGSZ, 
                          save=False, verbose=False)
    result = preds[0]
    boxes = result.boxes
    
    raw_dets = []
    if len(boxes) > 0:
        xyxy  = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        clses = boxes.cls.cpu().numpy()
        
        for i in range(len(xyxy)):
            raw_dets.append({
                'box': xyxy[i].tolist(),
                'conf': float(confs[i]),
                'cls': int(clses[i]),
            })
    
    # Step 2: Dilate
    dilated_dets = dilate_boxes(raw_dets, padding, img_w, img_h)
    
    # Step 3: Filter
    filtered_dets, filter_log = filter_detections(
        dilated_dets, img_w, img_h, ar_min, ar_max, min_area_pct, nested_iou
    )
    
    # Step 4: Crop
    img_np = np.array(img)
    crops = []
    for det in filtered_dets:
        x1, y1, x2, y2 = [int(round(c)) for c in det['box']]
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_w, x2)
        y2 = min(img_h, y2)
        
        if x2 > x1 and y2 > y1:
            crop = img_np[y1:y2, x1:x2]
            crops.append(Image.fromarray(crop))
        else:
            crops.append(None)
    
    return {
        'image_path': image_path,
        'img_w': img_w,
        'img_h': img_h,
        'raw_detections': raw_dets,
        'dilated_detections': dilated_dets,
        'filtered_detections': filtered_dets,
        'crops': crops,
        'filter_log': filter_log,
    }

# -------------------- Run Pipeline on All Test Images --------------------
print(f"\n Running full pipeline on {len(test_image_files)} test images...")

pipeline_results = {}
pipeline_start = time.time()

for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)
    pipeline_results[img_file] = run_pipeline(img_path, model_infer)

pipeline_elapsed = time.time() - pipeline_start

# -------------------- Summary --------------------
total_raw   = sum(len(r['raw_detections']) for r in pipeline_results.values())
total_final = sum(len(r['filtered_detections']) for r in pipeline_results.values())
total_crops = sum(len(r['crops']) for r in pipeline_results.values())

print(f"\n" + "-"*60)
print(f" PIPELINE SUMMARY")
print(f"-"*60)
print(f"   Images processed:    {len(pipeline_results)}")
print(f"   Raw detections:      {total_raw}")
print(f"   After dilation:      {total_raw} (same count, expanded boxes)")
print(f"   After filtering:     {total_final}")
print(f"   Table crops:         {total_crops}")
print(f"   Pipeline time:       {pipeline_elapsed:.1f}s ({pipeline_elapsed/max(len(pipeline_results),1)*1000:.0f}ms/image)")
print("="*60)

<a id='post-eval'></a>
## 12. Post-Processing Evaluation

Compare **raw predictions** vs. **post-processed predictions** (after dilation + filtering) against ground truth on the test set.

Custom IoU computation per image, plus aggregate mAP/F1 comparison to quantify the improvement from post-processing.

In [ ]:
# ============================================================
# Cell 12: Post-Processing Evaluation — Raw vs Filtered vs Ground Truth
# ============================================================

print("="*60)
print(" POST-PROCESSING EVALUATION")
print("="*60)

# -------------------- Load Ground Truth Labels --------------------
test_labels_dir = os.path.join(LABELS_DIR, 'test')

def load_yolo_gt(label_path, img_w, img_h):
    """Load YOLO ground truth and convert to [x1, y1, x2, y2] pixel coords."""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            
            # Convert to pixel xyxy
            w_px = w * img_w
            h_px = h * img_h
            x1 = xc * img_w - w_px / 2
            y1 = yc * img_h - h_px / 2
            x2 = x1 + w_px
            y2 = y1 + h_px
            
            boxes.append({
                'box': [x1, y1, x2, y2],
                'cls': cls,
            })
    return boxes

def match_predictions_to_gt(pred_boxes, gt_boxes, iou_threshold=0.5):
    """
    Match predicted boxes to ground truth using greedy IoU matching.
    
    Returns:
        tp: number of true positives (matched pairs with IoU >= threshold)
        fp: number of false positives (predictions without matching GT)
        fn: number of false negatives (GT without matching prediction)
        matched_ious: list of IoU values for matched pairs
    """
    if not pred_boxes and not gt_boxes:
        return 0, 0, 0, []
    if not pred_boxes:
        return 0, 0, len(gt_boxes), []
    if not gt_boxes:
        return 0, len(pred_boxes), 0, []
    
    # Compute IoU matrix
    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)))
    for i, pred in enumerate(pred_boxes):
        for j, gt in enumerate(gt_boxes):
            iou_matrix[i, j] = compute_iou_xyxy(pred['box'], gt['box'])
    
    # Greedy matching (highest IoU first)
    matched_pred = set()
    matched_gt = set()
    matched_ious = []
    
    while True:
        if iou_matrix.size == 0:
            break
        max_iou = iou_matrix.max()
        if max_iou < iou_threshold:
            break
        
        max_idx = np.unravel_index(iou_matrix.argmax(), iou_matrix.shape)
        pred_idx, gt_idx = max_idx
        
        matched_pred.add(pred_idx)
        matched_gt.add(gt_idx)
        matched_ious.append(max_iou)
        
        # Zero out matched rows/cols
        iou_matrix[pred_idx, :] = 0
        iou_matrix[:, gt_idx] = 0
    
    tp = len(matched_ious)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    
    return tp, fp, fn, matched_ious

# -------------------- Evaluate Raw vs Filtered --------------------
eval_records = []

for img_file in test_image_files:
    # Ground truth
    label_file = os.path.splitext(img_file)[0] + '.txt'
    label_path = os.path.join(test_labels_dir, label_file)
    
    img_path = os.path.join(test_images_dir, img_file)
    img = Image.open(img_path)
    img_w, img_h = img.size
    
    gt_boxes = load_yolo_gt(label_path, img_w, img_h)
    
    # Raw predictions
    raw_dets = raw_predictions.get(img_file, [])
    
    # Filtered predictions (from pipeline)
    result = pipeline_results.get(img_file, {})
    filt_dets = result.get('filtered_detections', [])
    
    # Match raw predictions to GT
    tp_raw, fp_raw, fn_raw, ious_raw = match_predictions_to_gt(raw_dets, gt_boxes)
    
    # Match filtered predictions to GT
    tp_filt, fp_filt, fn_filt, ious_filt = match_predictions_to_gt(filt_dets, gt_boxes)
    
    eval_records.append({
        'image': img_file,
        'n_gt': len(gt_boxes),
        'n_raw': len(raw_dets),
        'n_filtered': len(filt_dets),
        'tp_raw': tp_raw,
        'fp_raw': fp_raw,
        'fn_raw': fn_raw,
        'mean_iou_raw': np.mean(ious_raw) if ious_raw else 0.0,
        'tp_filt': tp_filt,
        'fp_filt': fp_filt,
        'fn_filt': fn_filt,
        'mean_iou_filt': np.mean(ious_filt) if ious_filt else 0.0,
    })

df_eval = pd.DataFrame(eval_records)

# -------------------- Aggregate Metrics --------------------
# Raw
total_tp_raw = df_eval['tp_raw'].sum()
total_fp_raw = df_eval['fp_raw'].sum()
total_fn_raw = df_eval['fn_raw'].sum()

p_raw = total_tp_raw / (total_tp_raw + total_fp_raw) if (total_tp_raw + total_fp_raw) > 0 else 0
r_raw = total_tp_raw / (total_tp_raw + total_fn_raw) if (total_tp_raw + total_fn_raw) > 0 else 0
f1_raw = 2 * p_raw * r_raw / (p_raw + r_raw) if (p_raw + r_raw) > 0 else 0
mean_iou_raw_all = df_eval['mean_iou_raw'].mean()

# Filtered
total_tp_filt = df_eval['tp_filt'].sum()
total_fp_filt = df_eval['fp_filt'].sum()
total_fn_filt = df_eval['fn_filt'].sum()

p_filt = total_tp_filt / (total_tp_filt + total_fp_filt) if (total_tp_filt + total_fp_filt) > 0 else 0
r_filt = total_tp_filt / (total_tp_filt + total_fn_filt) if (total_tp_filt + total_fn_filt) > 0 else 0
f1_filt = 2 * p_filt * r_filt / (p_filt + r_filt) if (p_filt + r_filt) > 0 else 0
mean_iou_filt_all = df_eval['mean_iou_filt'].mean()

# -------------------- Comparison Table --------------------
print("\n" + "-"*60)
print(" RAW vs POST-PROCESSED COMPARISON")
print("-"*60)

comparison_data = [
    ['Total Detections', f'{df_eval["n_raw"].sum()}', f'{df_eval["n_filtered"].sum()}'],
    ['True Positives',   f'{total_tp_raw}',           f'{total_tp_filt}'],
    ['False Positives',  f'{total_fp_raw}',           f'{total_fp_filt}'],
    ['False Negatives',  f'{total_fn_raw}',           f'{total_fn_filt}'],
    ['Precision',        f'{p_raw:.4f}',              f'{p_filt:.4f}'],
    ['Recall',           f'{r_raw:.4f}',              f'{r_filt:.4f}'],
    ['F1-Score',         f'{f1_raw:.4f}',             f'{f1_filt:.4f}'],
    ['Mean IoU',         f'{mean_iou_raw_all:.4f}',   f'{mean_iou_filt_all:.4f}'],
]

df_comparison = pd.DataFrame(comparison_data, columns=['Metric', 'Raw Predictions', 'Post-Processed'])
display(df_comparison.style.hide(axis='index'))

# -------------------- Delta --------------------
print(f"\n IMPROVEMENT FROM POST-PROCESSING:")
print(f"   Precision: {p_raw:.4f} → {p_filt:.4f} (Δ = {p_filt - p_raw:+.4f})")
print(f"   Recall:    {r_raw:.4f} → {r_filt:.4f} (Δ = {r_filt - r_raw:+.4f})")
print(f"   F1-Score:  {f1_raw:.4f} → {f1_filt:.4f} (Δ = {f1_filt - f1_raw:+.4f})")
print(f"   Mean IoU:  {mean_iou_raw_all:.4f} → {mean_iou_filt_all:.4f} (Δ = {mean_iou_filt_all - mean_iou_raw_all:+.4f})")

# -------------------- Per-Image Table --------------------
print("\n" + "-"*60)
print(" PER-IMAGE BREAKDOWN (first 20 images)")
print("-"*60)

display_eval_cols = ['image', 'n_gt', 'n_raw', 'n_filtered', 
                     'tp_filt', 'fp_filt', 'fn_filt', 'mean_iou_filt']
df_eval_display = df_eval[display_eval_cols].head(20).copy()
df_eval_display.columns = ['Image', 'GT Boxes', 'Raw Preds', 'Filtered Preds',
                           'TP', 'FP', 'FN', 'Mean IoU']

display(df_eval_display.style.format({'Mean IoU': '{:.4f}'}).hide(axis='index'))

print("="*60)

<a id='results-viz'></a>
## 13. Results Visualization

1. **Detection Overlay Grid:** Test images with GT (green), raw predictions (red dashed), and post-processed predictions (blue solid)
2. **Raw vs Dilated Crop Comparison:** Side-by-side showing the dilation effect on border preservation
3. **Confidence Distribution:** Histogram of prediction confidence scores

In [ ]:
# ============================================================
# Cell 13: Results Visualization
# ============================================================

print("="*60)
print(" RESULTS VISUALIZATION")
print("="*60)

# =====================================================================
# PLOT 1: Detection Overlay Grid (GT + Raw + Filtered)
# =====================================================================
print("\n Generating detection overlay grid...")

# Select images with detections for visualization
images_with_tables = [f for f in test_image_files 
                      if len(pipeline_results.get(f, {}).get('filtered_detections', [])) > 0]

# If few images have detections, include some without
if len(images_with_tables) < 9:
    remaining = [f for f in test_image_files if f not in images_with_tables]
    images_with_tables.extend(remaining[:9 - len(images_with_tables)])

sample_images = images_with_tables[:9]
n_samples = len(sample_images)

if n_samples > 0:
    n_cols = min(3, n_samples)
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    if n_samples == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_file in enumerate(sample_images):
        r, c = idx // n_cols, idx % n_cols
        ax = axes[r, c]
        
        img_path = os.path.join(test_images_dir, img_file)
        img = Image.open(img_path)
        img_w, img_h = img.size
        ax.imshow(img, cmap='gray')
        
        # Load and draw GT boxes (green)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_path = os.path.join(test_labels_dir, label_file)
        gt_boxes = load_yolo_gt(label_path, img_w, img_h)
        
        for gt in gt_boxes:
            x1, y1, x2, y2 = gt['box']
            rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                             edgecolor='#2ECC71', facecolor='none', linestyle='-',
                             label='GT' if gt == gt_boxes[0] else '')
            ax.add_patch(rect)
        
        # Draw raw predictions (red dashed)
        raw_dets = raw_predictions.get(img_file, [])
        for det in raw_dets:
            x1, y1, x2, y2 = det['box']
            rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                             edgecolor='#E74C3C', facecolor='none', linestyle='--',
                             label='Raw' if det == raw_dets[0] else '')
            ax.add_patch(rect)
        
        # Draw filtered predictions (blue solid)
        result = pipeline_results.get(img_file, {})
        filt_dets = result.get('filtered_detections', [])
        for det in filt_dets:
            x1, y1, x2, y2 = det['box']
            rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2.5,
                             edgecolor='#3498DB', facecolor='none', linestyle='-',
                             label='Filtered' if det == filt_dets[0] else '')
            ax.add_patch(rect)
            ax.text(x1, y1-8, f'{det["conf"]:.2f}', color='#3498DB', fontsize=8,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        
        n_gt = len(gt_boxes)
        n_filt = len(filt_dets)
        ax.set_title(f'{img_file}\nGT: {n_gt} | Filtered: {n_filt}', fontsize=9)
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(n_samples, n_rows * n_cols):
        r, c = idx // n_cols, idx % n_cols
        axes[r, c].axis('off')
    
    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='#2ECC71', linewidth=2, linestyle='-', label='Ground Truth'),
        Line2D([0], [0], color='#E74C3C', linewidth=1.5, linestyle='--', label='Raw Prediction'),
        Line2D([0], [0], color='#3498DB', linewidth=2.5, linestyle='-', label='Post-Processed'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11,
               bbox_to_anchor=(0.5, -0.02))
    
    fig.suptitle('Detection Results: Ground Truth vs Raw vs Post-Processed',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# =====================================================================
# PLOT 2: Raw vs Dilated Crop Comparison
# =====================================================================
print("\n Generating raw vs dilated crop comparison...")

# Find images with both raw and filtered detections that have 'box_raw'
crop_comparison_images = []
for img_file, result in pipeline_results.items():
    filt_dets = result.get('filtered_detections', [])
    if filt_dets and 'box_raw' in filt_dets[0]:
        crop_comparison_images.append(img_file)
    if len(crop_comparison_images) >= 4:
        break

if crop_comparison_images:
    n_compare = len(crop_comparison_images)
    fig, axes = plt.subplots(n_compare, 2, figsize=(10, 4 * n_compare))
    if n_compare == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_file in enumerate(crop_comparison_images):
        result = pipeline_results[img_file]
        img = Image.open(result['image_path'])
        img_np = np.array(img)
        img_w, img_h = img.size
        
        det = result['filtered_detections'][0]  # First detection
        
        # Raw crop (before dilation)
        raw_box = det.get('box_raw', det['box'])
        rx1, ry1, rx2, ry2 = [int(round(c)) for c in raw_box]
        rx1, ry1 = max(0, rx1), max(0, ry1)
        rx2, ry2 = min(img_w, rx2), min(img_h, ry2)
        raw_crop = img_np[ry1:ry2, rx1:rx2]
        
        # Dilated crop (after dilation)
        dx1, dy1, dx2, dy2 = [int(round(c)) for c in det['box']]
        dx1, dy1 = max(0, dx1), max(0, dy1)
        dx2, dy2 = min(img_w, dx2), min(img_h, dy2)
        dil_crop = img_np[dy1:dy2, dx1:dx2]
        
        axes[idx, 0].imshow(raw_crop, cmap='gray')
        axes[idx, 0].set_title(f'Raw Crop ({rx2-rx1}x{ry2-ry1}px)', fontsize=10)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(dil_crop, cmap='gray')
        axes[idx, 1].set_title(f'Dilated Crop +{BOX_PADDING_PX}px ({dx2-dx1}x{dy2-dy1}px)', fontsize=10)
        axes[idx, 1].axis('off')
    
    fig.suptitle(f'Raw vs Dilated Crops (padding={BOX_PADDING_PX}px)',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("   No crop comparison available (no detections with box_raw).")

# =====================================================================
# PLOT 3: Confidence Distribution
# =====================================================================
print("\n Generating confidence distribution...")

all_raw_confs = [d['conf'] for dets in raw_predictions.values() for d in dets]
all_filt_confs = [d['conf'] for r in pipeline_results.values() 
                  for d in r.get('filtered_detections', [])]

if all_raw_confs or all_filt_confs:
    fig, ax = plt.subplots(figsize=(10, 4))
    
    if all_raw_confs:
        ax.hist(all_raw_confs, bins=30, alpha=0.5, color='#E74C3C', 
                edgecolor='black', label=f'Raw (n={len(all_raw_confs)})')
    if all_filt_confs:
        ax.hist(all_filt_confs, bins=30, alpha=0.5, color='#3498DB',
                edgecolor='black', label=f'Filtered (n={len(all_filt_confs)})')
    
    ax.axvline(x=CONF_THRESH, color='green', linestyle='--', linewidth=2,
               label=f'Conf threshold ({CONF_THRESH})')
    ax.set_xlabel('Confidence Score', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("   All visualization plots generated.")
print("="*60)

<a id='export'></a>
## 14. Export Results

Save:
1. **Cropped table images** to `/kaggle/working/cropped_tables/`
2. **Detection results JSON** with all boxes, scores, and metadata
3. **Final summary report** with all evaluation metrics

In [ ]:
# ============================================================
# Cell 14: Export Results
# ============================================================

print("="*60)
print(" EXPORT RESULTS")
print("="*60)

# =====================================================================
# 1. Save Cropped Table Images
# =====================================================================
print(f"\n Saving cropped tables to: {CROPS_DIR}")

if os.path.exists(CROPS_DIR):
    shutil.rmtree(CROPS_DIR)
os.makedirs(CROPS_DIR, exist_ok=True)

crop_count = 0

for img_file, result in pipeline_results.items():
    crops = result.get('crops', [])
    filt_dets = result.get('filtered_detections', [])
    
    for i, (crop, det) in enumerate(zip(crops, filt_dets)):
        if crop is None:
            continue
        
        base_name = os.path.splitext(img_file)[0]
        crop_name = f"{base_name}_table_{i+1}.jpg"
        crop_path = os.path.join(CROPS_DIR, crop_name)
        
        crop.save(crop_path, quality=95)
        crop_count += 1

print(f"   Saved {crop_count} cropped table images.")

# =====================================================================
# 2. Save Detection Results JSON
# =====================================================================
results_json_path = os.path.join(OUTPUT_DIR, 'detection_results.json')

export_results = []
for img_file, result in pipeline_results.items():
    img_entry = {
        'image_name': img_file,
        'img_w': result['img_w'],
        'img_h': result['img_h'],
        'n_raw_detections': len(result['raw_detections']),
        'n_filtered_detections': len(result['filtered_detections']),
        'detections': []
    }
    
    for i, det in enumerate(result['filtered_detections']):
        img_entry['detections'].append({
            'table_id': i + 1,
            'box_xyxy': [round(c, 2) for c in det['box']],
            'box_raw_xyxy': [round(c, 2) for c in det.get('box_raw', det['box'])],
            'confidence': round(det['conf'], 4),
            'class': det['cls'],
            'class_name': CLASS_NAMES[det['cls']] if det['cls'] < len(CLASS_NAMES) else 'unknown',
        })
    
    export_results.append(img_entry)

with open(results_json_path, 'w') as f:
    json.dump(export_results, f, indent=2)

print(f"   Detection results saved to: {results_json_path}")

# =====================================================================
# 3. Final Summary Report
# =====================================================================
print("\n" + "="*70)
print(" PHASE 3 & 4 FINAL SUMMARY REPORT")
print("="*70)

print("\n" + "-"*70)
print(" PHASE 3: TRAINING SUMMARY")
print("-"*70)

training_summary = [
    ['Model',                   MODEL_VARIANT],
    ['Resolution',              f'{IMGSZ}×{IMGSZ} (letterbox)'],
    ['Epochs Trained',          f'{len(df_metrics)} / {EPOCHS}'],
    ['Early Stopping',          f'{"Yes (patience={})".format(PATIENCE) if len(df_metrics) < EPOCHS else "No (all epochs completed)"}'],
    ['Optimizer',               f'{OPTIMIZER} (lr={LR0}, wd={WEIGHT_DECAY})'],
    ['Batch Size',              str(BATCH_SIZE)],
    ['Training Time',           f'{train_elapsed/60:.1f} min'],
]

if 'mAP@0.5' in df_metrics.columns:
    best_idx = df_metrics['mAP@0.5:0.95'].idxmax()
    best = df_metrics.loc[best_idx]
    training_summary.extend([
        ['', ''],
        ['Best Epoch',              str(int(best.get('Epoch', best_idx)))],
        ['Best mAP@0.5',           f'{best.get("mAP@0.5", 0):.4f}'],
        ['Best mAP@0.5:0.95',     f'{best.get("mAP@0.5:0.95", 0):.4f}'],
        ['Best F1-Score',          f'{best.get("F1-Score", 0):.4f}'],
    ])

df_train_summary = pd.DataFrame(training_summary, columns=['Metric', 'Value'])
display(df_train_summary.style.hide(axis='index'))

print("\n" + "-"*70)
print(" PHASE 3: TEST SET RESULTS")
print("-"*70)

test_results_display = [
    ['Precision',       f'{test_precision:.4f}'],
    ['Recall',          f'{test_recall:.4f}'],
    ['F1-Score',        f'{test_f1:.4f}'],
    ['mAP@0.5',        f'{test_map50:.4f}'],
    ['mAP@0.5:0.95',   f'{test_map5095:.4f}'],
]
df_test_display = pd.DataFrame(test_results_display, columns=['Metric', 'Value'])
display(df_test_display.style.hide(axis='index'))

print("\n" + "-"*70)
print(" PHASE 4: POST-PROCESSING SUMMARY")
print("-"*70)

pp_summary = [
    ['Box Padding',          f'{BOX_PADDING_PX}px'],
    ['AR Filter Range',      f'[{AR_MIN}, {AR_MAX}]'],
    ['Min Area Filter',      f'{MIN_AREA_PCT*100:.0f}% of image'],
    ['Nested IoU Threshold', f'{NESTED_IOU}'],
    ['', ''],
    ['Raw Detections',       f'{total_raw}'],
    ['Post-Processed',       f'{total_final}'],
    ['Removed by Filters',   f'{total_raw - total_final}'],
    ['Table Crops Saved',    f'{crop_count}'],
    ['', ''],
    ['Precision (post)',     f'{p_filt:.4f}'],
    ['Recall (post)',        f'{r_filt:.4f}'],
    ['F1-Score (post)',      f'{f1_filt:.4f}'],
    ['Mean IoU (post)',      f'{mean_iou_filt_all:.4f}'],
]

df_pp_summary = pd.DataFrame(pp_summary, columns=['Metric', 'Value'])
display(df_pp_summary.style.hide(axis='index'))

# =====================================================================
# Completion Checklist
# =====================================================================
print("\n" + "="*70)
print(" COMPLETION CHECKLIST")
print("="*70)

checklist = [
    ("Phase 3: YOLOv10-M model trained",          os.path.exists(BEST_WEIGHTS)),
    ("Phase 3: Per-epoch metrics displayed",       len(df_metrics) > 0),
    ("Phase 3: Training curves plotted",           True),
    ("Phase 3: Test set evaluated",                test_map50 > 0),
    ("Phase 4: Inference on test set",             total_raw >= 0),
    ("Phase 4: Box dilation applied",              BOX_PADDING_PX > 0),
    ("Phase 4: Heuristic filters applied",         total_filter_log['input'] >= 0),
    ("Phase 4: End-to-end pipeline verified",      len(pipeline_results) > 0),
    ("Phase 4: Raw vs post-processed compared",    len(df_eval) > 0),
    ("Phase 4: Results visualized",                True),
    ("Phase 4: Crops exported",                    crop_count >= 0),
    ("Phase 4: Detection JSON exported",           os.path.exists(results_json_path)),
]

all_passed = True
for task, passed in checklist:
    status = "OK" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"   {status}  {task}")

print("\n" + "="*70)
if all_passed:
    print(" ALL CHECKS PASSED")
else:
    print(" SOME CHECKS FAILED — REVIEW ABOVE")
print("="*70)

print(f"\n Output Files:")
print(f"   Best weights: {BEST_WEIGHTS}")
print(f"   Cropped tables: {CROPS_DIR} ({crop_count} images)")
print(f"   Detection JSON: {results_json_path}")
print(f"   Training results: {RESULTS_CSV}")